# Evaluación final Nemotron — MediaSpeech ES

Esta notebook es **independiente** de los scripts de testing y sweeps. Ejecuta una sola evaluación fija sobre los 2.507 clips con referencia humana:

1. transcripción offline de archivo completo;
2. streaming cache-aware acelerado usando la misma `NemotronSession` de producción, sin `sleep`;
3. el replay físico de una hora queda deliberadamente fuera de esta notebook.

Configuración congelada: `es-ES`, lookahead `560 ms` (`[56,6]`), `stop_history_eou=600 ms`, residuo `2`, RNNT `greedy_batch`, float32 + AMP. Los checkpoints se guardan en Drive cada 25 clips; volver a ejecutar **Run all** reanuda sin repetir clips exitosos. No usa placa, bridge, ngrok ni firmware.


## 1. GPU y Drive


In [ ]:
import importlib, json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

import torch
from google.colab import drive

print('python:', platform.python_version())
print('torch :', torch.__version__, 'cuda:', torch.version.cuda)
print('gpu   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Activá Runtime -> Change runtime type -> GPU'
drive.mount('/content/drive')


## 2. Rutas persistentes

Subí el tar original, sin extraer, a `MyDrive/TESIS/stt_benchmarks/mediaspeech_es/v1.1/ES.tgz`. La celda también busca dos ubicaciones compatibles por si ya lo dejaste en `Tesis-subtitles`.


In [ ]:
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/TESIS')
LEGACY_DRIVE_ROOT = Path('/content/drive/MyDrive/Tesis-subtitles')
DATASET_ARCHIVE_OVERRIDE = None  # Path(...) sólo si lo guardaste en otro lugar

archive_candidates = ([Path(DATASET_ARCHIVE_OVERRIDE)] if DATASET_ARCHIVE_OVERRIDE else []) + [
    DRIVE_PROJECT_ROOT / 'stt_benchmarks' / 'mediaspeech_es' / 'v1.1' / 'ES.tgz',
    LEGACY_DRIVE_ROOT / 'stt_benchmarks' / 'mediaspeech_es' / 'v1.1' / 'ES.tgz',
    LEGACY_DRIVE_ROOT / 'ES.tgz',
]
DATASET_ARCHIVE = next((path for path in archive_candidates if path.is_file()), None)
assert DATASET_ARCHIVE is not None, 'No encontré ES.tgz. Revisá archive_candidates.'

CACHE_ROOT = LEGACY_DRIVE_ROOT / 'nemotron' / 'cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['NEMO_CACHE_DIR'] = str(CACHE_ROOT / 'nemo')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')

def colab_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

hf_token = colab_secret('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
print('dataset:', DATASET_ARCHIVE)
print('cache  :', CACHE_ROOT)


## 3. Traer la rama `dev/nemotron`


In [ ]:
REPO_URL = 'https://github.com/Nacholazabal/subtitle_overlay_fw.git'
REPO_BRANCH = 'dev/nemotron'
REPO_DIR = Path('/content/subtitle_overlay_fw')

remote = subprocess.run(['git', 'ls-remote', '--heads', REPO_URL, REPO_BRANCH], capture_output=True, text=True)
assert remote.stdout.strip(), f'No existe {REPO_BRANCH} en GitHub; pusheá esta implementación primero'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
PROJECT_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('project commit:', PROJECT_COMMIT)


## 4. Verificar los archivos nuevos e importar el paquete correcto


In [ ]:
REQUIRED = [
    'server/evaluation/dataset_manifest.py',
    'server/evaluation/dataset.py',
    'server/runtime/nemotron.py',
    'server/evaluation/probe.py',
]
missing = [name for name in REQUIRED if not (REPO_DIR / name).is_file()]
assert not missing, f'La rama remota todavía no contiene: {missing}'
repo_path = str(REPO_DIR)
sys.path[:] = [path for path in sys.path if path != repo_path]
sys.path.insert(0, repo_path)
for module_name in [name for name in list(sys.modules) if name == 'server' or name.startswith('server.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()
import server as project_server
assert Path(project_server.__file__).resolve() == (REPO_DIR / 'server' / '__init__.py').resolve()
print('project package:', project_server.__file__)


## 5. NeMo fijado y dependencias


In [ ]:
from server.runtime.nemotron import NEMO_COMMIT, NEMO_REPO

NEMO_DIR = Path('/content/NeMo')
if not NEMO_DIR.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', NEMO_REPO, str(NEMO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(NEMO_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(NEMO_DIR), 'checkout', '-q', NEMO_COMMIT], check=True)
resolved = subprocess.check_output(['git', '-C', str(NEMO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert resolved == NEMO_COMMIT
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libsndfile1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'Cython', 'packaging', 'soundfile'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{NEMO_DIR}[asr]'], check=True)
if str(NEMO_DIR) not in sys.path:
    sys.path.insert(0, str(NEMO_DIR))
importlib.invalidate_caches()
import nemo.collections.asr
from nemo.collections.asr.inference.factory.pipeline_builder import PipelineBuilder
print('NeMo ready at', resolved)


## 6. Verificar, copiar y extraer MediaSpeech localmente

Drive conserva sólo el tar y los resultados. Los 5.014 archivos se extraen en `/content`, no sobre Drive.


In [ ]:
from server.evaluation.dataset_manifest import (
    ARCHIVE_SHA256, build_mediaspeech_manifest, discover_dataset_root,
    safe_extract_archive, verify_archive, write_manifest,
)

drive_archive = verify_archive(DATASET_ARCHIVE)
print('Drive archive verified:', drive_archive)
LOCAL_DATA_ROOT = Path('/content/datasets/mediaspeech_es')
LOCAL_ARCHIVE = Path('/content/datasets/ES.tgz')
LOCAL_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
local_archive_ok = False
if LOCAL_ARCHIVE.is_file():
    try:
        local_archive_ok = verify_archive(LOCAL_ARCHIVE)['sha256'] == ARCHIVE_SHA256
    except Exception:
        local_archive_ok = False
if not local_archive_ok:
    shutil.copy2(DATASET_ARCHIVE, LOCAL_ARCHIVE)
verify_archive(LOCAL_ARCHIVE)
extract_marker = LOCAL_DATA_ROOT / '.archive_sha256'
extraction_complete = extract_marker.is_file() and extract_marker.read_text().strip() == ARCHIVE_SHA256
if not extraction_complete:
    if LOCAL_DATA_ROOT.exists():
        shutil.rmtree(LOCAL_DATA_ROOT)
    safe_extract_archive(LOCAL_ARCHIVE, LOCAL_DATA_ROOT)
    extract_marker.write_text(ARCHIVE_SHA256 + '\n', encoding='utf-8')
DATASET_ROOT = discover_dataset_root(LOCAL_DATA_ROOT)
clips = build_mediaspeech_manifest(DATASET_ROOT)
assert len(clips) == 2507
print('dataset root:', DATASET_ROOT)
print('paired clips:', len(clips))


## 7. Cargar una sola vez el modelo con la configuración congelada


In [ ]:
from server.runtime.nemotron import SharedNemotronModel
from server.evaluation.dataset import EVALUATION_ID, fixed_nemotron_config

fixed_config = fixed_nemotron_config()
print(json.dumps(fixed_config.as_effective_config(), indent=2, ensure_ascii=False))
model_load_started = time.monotonic()
shared_model = SharedNemotronModel(fixed_config)
print('model loaded in', round(time.monotonic() - model_load_started, 2), 's')
print(json.dumps(shared_model.provenance(), indent=2, ensure_ascii=False))


## 8. Abrir/reanudar la evaluación

La carpeta no lleva timestamp a propósito: representa esta única evaluación fija. Si cambia el commit, modelo, corpus o configuración, el runner aborta antes de mezclar resultados.


In [ ]:
from server.evaluation.dataset import NemotronDatasetEvaluator

EVAL_OUTPUT = DRIVE_PROJECT_ROOT / 'stt_evaluations' / 'mediaspeech_es' / EVALUATION_ID
EVAL_OUTPUT.mkdir(parents=True, exist_ok=True)
evaluator = NemotronDatasetEvaluator(
    shared_model, DATASET_ROOT, clips, EVAL_OUTPUT,
    project_commit=PROJECT_COMMIT, config=fixed_config, checkpoint_every=25,
)
write_manifest(EVAL_OUTPUT / 'manifest.jsonl', clips)  # identity already accepted
print('evaluation:', EVAL_OUTPUT)
print('fingerprint:', evaluator.identity['fingerprint'])


## 9. Fase 1 — todos los clips offline


In [ ]:
offline_progress = evaluator.run_offline()
print(json.dumps(offline_progress, indent=2, ensure_ascii=False))
if offline_progress['status'] != 'complete':
    raise RuntimeError('Offline quedó incompleto. Corregí el error y reejecutá Run all: se reanuda.')


## 10. Fase 2 — todos los clips streaming acelerado


In [ ]:
streaming_progress = evaluator.run_streaming()
print(json.dumps(streaming_progress, indent=2, ensure_ascii=False))
if streaming_progress['status'] != 'complete':
    raise RuntimeError('Streaming quedó incompleto. Reejecutá Run all para reanudar fallos.')


## 11. Reporte final persistido en Drive


In [ ]:
summary = evaluator.write_reports()
assert summary['status'] == 'complete'
print((EVAL_OUTPUT / 'report.md').read_text(encoding='utf-8'))
print('RESULTADOS GUARDADOS EN:', EVAL_OUTPUT)
